<a href="https://colab.research.google.com/github/Tmiller68/machine-learning-fundamentals/blob/main/Day_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded=files.upload()

In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split

column_names = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]



df = pd.read_csv(
    "adult.data",
    names=column_names,
    header=None)
df = df.replace(" ?", np.nan)
df = df.dropna()

In [ ]:
X=df.drop("income", axis=1)
Y=df["income"].str.strip()
label_encoder=LabelEncoder()
Y=label_encoder.fit_transform(Y)

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

In [ ]:
numeric_columns=X.select_dtypes(
    include=["int64", "float64"]
).columns
categorical_columns=X.select_dtypes(
    include=["object"]
).columns

In [ ]:
print(numeric_columns)
print(categorical_columns)

In [ ]:
preprocessor = ColumnTransformer([
    (
        "numeric",
        StandardScaler(),
        numeric_columns
    ),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_columns
    )
])

In [ ]:
adult_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

In [ ]:
pipeline_scores = cross_val_score(
    adult_pipeline,
    X,
    Y,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

In [ ]:
print("Pipeline fold scores:")
print(pipeline_scores)

print(
    "Pipeline mean accuracy:",
    pipeline_scores.mean()
)

print(
    "Pipeline standard deviation:",
    pipeline_scores.std()
)

In [ ]:
leaky_preprocessor = ColumnTransformer([
    (
        "numeric",
        StandardScaler(),
        numeric_columns
    ),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_columns
    )
])
X=leaky_preprocessor.fit_transform(X)

In [ ]:
leaky_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

leaky_scores = cross_val_score(
    leaky_model,
    X,
    Y,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

In [ ]:
print("Leaked fold scores:")
print(leaky_scores)

print(
    "Leaked mean accuracy:",
    leaky_scores.mean()
)

print(
    "Leaked standard deviation:",
    leaky_scores.std()
)

In [ ]:
day4_results = pd.DataFrame({
    "Method": [
        "Leakage-Free Pipeline",
        "Preprocessing Before CV"
    ],
    "Mean CV Accuracy": [
        pipeline_scores.mean(),
        leaky_scores.mean()
    ],
    "Accuracy STD": [
        pipeline_scores.std(),
        leaky_scores.std()
    ]
})

day4_results

In [ ]:
day4_results["Mean CV Accuracy"] = (
    day4_results["Mean CV Accuracy"].round(4)
)

day4_results["Accuracy STD"] = (
    day4_results["Accuracy STD"].round(4)
)

day4_results

Day 5

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer([
    (
        "numeric",
        StandardScaler(),
        numeric_columns
    ),
    (
        "categorical",
        OneHotEncoder(
            handle_unknown="ignore"
        ),
        categorical_columns
    )
])

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

random_forest_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        )
    )
])

In [ ]:
parameter_distributions = {
    "classifier__n_estimators": [
        100,
        200,
        300,
        500
    ],

    "classifier__max_depth": [
        None,
        5,
        10,
        15,
        20
    ],

    "classifier__min_samples_leaf": [
        1,
        2,
        4,
        8
    ]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=random_forest_pipeline,
    param_distributions=parameter_distributions,
    n_iter=12,
    scoring="accuracy",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True
)

In [ ]:
random_search.fit(
    X_train,
    Y_train
)

In [ ]:
print("Best parameters:")
print(random_search.best_params_)

print("\nBest cross-validation accuracy:")
print(random_search.best_score_)

In [ ]:
best_random_forest = random_search.best_estimator_

In [ ]:
from sklearn.metrics import accuracy_score

test_predictions = best_random_forest.predict(X_test)

test_accuracy = accuracy_score(
    Y_test,
    test_predictions
)

print("Final test accuracy:", test_accuracy)

In [ ]:
from sklearn.inspection import permutation_importance
permutation_results = permutation_importance(
    best_random_forest,
    X_test,
    Y_test,
    scoring="accuracy",
    n_repeats=10,
    random_state=42,
    n_jobs=-1)

In [ ]:
importance_table = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance Mean": permutation_results.importances_mean,
    "Importance STD": permutation_results.importances_std
})

importance_table = importance_table.sort_values(
    by="Importance Mean",
    ascending=False
)

importance_table

In [ ]:
importance_table.head(10)

In [ ]:
import matplotlib.pyplot as plt

top_features = importance_table.head(10).sort_values(
    by="Importance Mean",
    ascending=True
)

plt.figure(figsize=(9, 6))

plt.barh(
    top_features["Feature"],
    top_features["Importance Mean"],
    xerr=top_features["Importance STD"]
)

plt.xlabel("Decrease in Test Accuracy")
plt.ylabel("Feature")
plt.title("Permutation Importance: Best Random Forest")
plt.tight_layout()
plt.show()

In [ ]:
random_forest_classifier = (
    best_random_forest.named_steps["classifier"]
)

impurity_importances = (
    random_forest_classifier.feature_importances_
)

I also finished the comparison between impurity-based feature importance and permutation importance. Impurity importance measures how much a feature helps a tree-based model make useful splits, while permutation importance measures how much the model’s performance decreases when a feature is randomly shuffled. I compared the rankings from both methods to see which features were consistently important and to understand why the two methods may give different results.